<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/11-capstone/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 11: Solutions

Try the exercises first. The notes after each solution explain the design choices and what goes wrong in real projects without them.

In [1]:
# Setup: run this cell once before you start.
# It installs FastAPI and httpx if they're missing, and defines the check functions used below.

import importlib.util

_missing = [pkg for pkg in ["fastapi", "httpx"] if importlib.util.find_spec(pkg) is None]
if _missing:
    _packages = " ".join(_missing)
    %pip install -q {_packages}

import math

import numpy as np
import pandas as pd
from fastapi.testclient import TestClient
from pydantic import ValidationError
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _as_list(series):
    """Series values as a plain list, with every kind of missing value shown as None."""
    return [None if pd.isna(v) else v for v in series]


def check_snake_case(f):
    _scenarios("snake_case", [
        ("spaces and units", lambda: f("Culmen Length (mm)"), "culmen_length_mm", None),
        ("digits and slashes", lambda: f("Delta 15 N (o/oo)"), "delta_15_n_o_oo", "replace each run of other characters with one _"),
        ("camelCase", lambda: f("studyName"), "study_name", "insert _ between a lowercase letter or digit and a capital first"),
        ("a run of capitals stays together", lambda: f("Individual ID"), "individual_id", None),
        ("surrounding spaces", lambda: f("  Body Mass (g) "), "body_mass_g", "strip leading and trailing _"),
        ("already snake_case", lambda: f("body_mass_g"), "body_mass_g", None),
        ("a digit before a capital", lambda: f("delta15N"), "delta15_n", None),
    ])


def check_clean_species(f):
    raw = pd.Series(["Adelie Penguin (Pygoscelis adeliae)", "Gentoo penguin (Pygoscelis papua)",
                     "Chinstrap penguin (Pygoscelis antarctica)", "  adelie penguin "], index=[10, 11, 12, 13])
    _scenarios("clean_species", [
        ("full names to short names", lambda: _as_list(f(raw)), ["Adelie", "Gentoo", "Chinstrap", "Adelie"],
         "strip, split, take the first word, capitalise"),
        ("keeps the index", lambda: f(raw).index.tolist(), [10, 11, 12, 13], None),
        ("an unknown species raises", lambda: f(pd.Series(["Emperor penguin (Aptenodytes forsteri)"])), Raises(ValueError),
         "check the cleaned values are all in the list of three species"),
        ("a missing value raises", lambda: f(pd.Series(["Adelie Penguin", None])), Raises(ValueError), None),
    ])


def check_clean_sex(f):
    raw = pd.Series(["MALE", "FEMALE", " male ", "Female", ".", "", "unknown", None, np.nan], index=range(5, 14))
    _scenarios("clean_sex", [
        ("cleans valid values; everything else is missing", lambda: _as_list(f(raw)),
         ["male", "female", "male", "female", None, None, None, None, None],
         "strip and lowercase, then keep only 'male' and 'female' (Series.where)"),
        ("keeps the index", lambda: f(raw).index.tolist(), list(range(5, 14)), None),
        ("doesn't turn missing values into the text 'nan'", lambda: "nan" in [str(v).lower() for v in f(raw).dropna()], False,
         "astype(str) turns NaN into 'nan'; use the .str methods directly"),
    ])


def check_out_of_range(f):
    df = pd.DataFrame({"mass": [3700.0, 37.0, np.nan, 9000.0], "depth": [18.0, 175.0, 17.0, 5.0]}, index=["a", "b", "c", "d"])
    limits = {"mass": (2000, 7000), "depth": (10, 25)}
    expected = pd.DataFrame({"row": ["b", "b", "d", "d"], "column": ["mass", "depth", "mass", "depth"],
                             "value": [37.0, 175.0, 9000.0, 5.0]})
    _scenarios("out_of_range", [
        ("finds every value outside its limits, in order", lambda: f(df, limits), expected,
         "one line per bad value, sorted by row then by the column's position in limits"),
        ("limits are inclusive", lambda: len(f(pd.DataFrame({"x": [1.0, 2.0, 3.0]}), {"x": (1, 3)})), 0, None),
        ("missing values are skipped", lambda: len(f(pd.DataFrame({"x": [np.nan]}), {"x": (0, 1)})), 0, None),
        ("nothing wrong: empty with the right columns", lambda: list(f(df.loc[["a"]], limits).columns),
         ["row", "column", "value"], "return pd.DataFrame(columns=['row', 'column', 'value'])"),
    ])


def check_make_preprocessor(f):
    train = pd.DataFrame({"mass": [3000.0, 3500.0, np.nan, 5500.0], "depth": [15.0, 16.0, 17.0, 18.0],
                          "island": ["Biscoe", "Dream", np.nan, "Biscoe"], "notes": ["a", "b", "c", "d"]})
    new = pd.DataFrame({"mass": [4000.0], "depth": [16.5], "island": ["Torgersen"], "notes": ["x"]})

    filled = np.array([3000.0, 3500.0, 3500.0, 5500.0])         # the median of the known values is 3500
    median_filled = np.round((filled - filled.mean()) / filled.std(), 6).tolist()

    def unfitted():
        pre = f(["mass", "depth"], ["island"])
        return isinstance(pre, ColumnTransformer) and not hasattr(pre, "transformers_")

    def output():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return (type(Z).__name__, Z.shape, bool(np.isnan(Z).any()))

    def scaled():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return (np.round(Z[:, 0], 6).tolist(), bool(np.allclose(Z[:, :2].mean(axis=0), 0)))

    def onehot():
        Z = f(["mass", "depth"], ["island"]).fit_transform(train)
        return Z[:, 2:].tolist()

    def unknown():
        pre = f(["mass", "depth"], ["island"]).fit(train)
        return pre.transform(new)[0, 2:].tolist()

    _scenarios("make_preprocessor", [
        ("returns an unfitted ColumnTransformer", unfitted, True, "build it and return it; don't call fit"),
        ("dense array, extra columns dropped, no missing values", output, ("ndarray", (4, 4), False),
         "OneHotEncoder(sparse_output=False); imputers before the scaler and encoder"),
        ("numeric: median fill, then standardised", scaled, (median_filled, True),
         "SimpleImputer(strategy='median') then StandardScaler()"),
        ("categorical: most frequent fill, then one-hot", onehot,
         [[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0]], "SimpleImputer(strategy='most_frequent') before the encoder"),
        ("unseen category becomes all zeros", unknown, [0.0, 0.0], "OneHotEncoder(handle_unknown='ignore')"),
    ])


def check_compare_models(f):
    X, y = load_breast_cancer(return_X_y=True)
    X, y = X[:200], y[:200]
    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    models = {"baseline": DummyClassifier(),
              "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
              "knn": make_pipeline(StandardScaler(), KNeighborsClassifier())}
    expected_scores = {name: cross_val_score(m, X, y, cv=cv) for name, m in models.items()}

    def table():
        result = f(models, X, y, cv)
        return (list(result.columns), list(result.index)[-1], bool(result["mean"].is_monotonic_decreasing))

    def values():
        result = f(models, X, y, cv)
        return [(round(result.loc[n, "mean"], 6), round(result.loc[n, "std"], 6)) for n in models]

    _scenarios("compare_models", [
        ("columns, the baseline last, sorted best first", table, (["mean", "std"], "baseline", True),
         "sort_values('mean', ascending=False)"),
        ("means and standard deviations of the fold scores", values,
         [(round(s.mean(), 6), round(s.std(), 6)) for s in expected_scores.values()],
         "cross_val_score(model, X, y, cv=cv) for each model, with the cv passed in"),
        ("models are left unfitted", lambda: (f(models, X, y, cv), hasattr(models["knn"][-1], "classes_"))[1], False,
         "cross_val_score fits clones, not the originals"),
    ])


def check_selective_accuracy(f):
    y = np.array([1, 0, 1, 0, 1, 0])
    p = np.array([0.95, 0.10, 0.55, 0.52, 0.25, 0.80])
    _scenarios("selective_accuracy", [
        ("margin 0: always answers", lambda: f(y, p, 0.0), {"coverage": 1.0, "accuracy": 3 / 6, "n_covered": 6},
         "predicted class is 1 when p >= 0.5"),
        ("margin 0.2: abstains near 0.5", lambda: f(y, p, 0.2), {"coverage": 4 / 6, "accuracy": 2 / 4, "n_covered": 4},
         "covered when abs(p - 0.5) >= margin"),
        ("margin 0.4", lambda: f(y, p, 0.4), {"coverage": 2 / 6, "accuracy": 1.0, "n_covered": 2}, None),
        ("nothing covered: accuracy is nan", lambda: math.isnan(f(y, p, 0.49)["accuracy"]) and f(y, p, 0.49)["n_covered"] == 0,
         True, "return nan rather than dividing by zero"),
        ("n_covered is an int", lambda: type(f(y, p, 0.2)["n_covered"]), int, "int(covered.sum())"),
    ])


def check_PenguinIn(f):
    good = {"species": "Gentoo", "bill_length_mm": 46.5, "bill_depth_mm": 14.8, "flipper_length_mm": 217, "body_mass_g": 5200}

    def make(**changes):
        values = {**good, **changes}
        values = {k: v for k, v in values.items() if v is not ...}
        return f(**values)

    _scenarios("PenguinIn", [
        ("a valid penguin", lambda: make().model_dump(), {**good, "flipper_length_mm": 217.0, "body_mass_g": 5200.0}, None),
        ("numbers sent as text are converted", lambda: make(bill_length_mm="46.5").bill_length_mm, 46.5, None),
        ("limits are inclusive", lambda: make(body_mass_g=7000).body_mass_g, 7000.0, "Field(ge=2000, le=7000)"),
        ("unknown species is rejected", lambda: make(species="Emperor"), Raises(ValidationError),
         "species: Literal['Adelie', 'Chinstrap', 'Gentoo']"),
        ("body mass out of range is rejected", lambda: make(body_mass_g=37), Raises(ValidationError), None),
        ("bill depth out of range is rejected", lambda: make(bill_depth_mm=175), Raises(ValidationError), None),
        ("a missing field is rejected", lambda: make(flipper_length_mm=...), Raises(ValidationError), None),
        ("an unknown field is rejected", lambda: make(body_mass=5200), Raises(ValidationError),
         "model_config = ConfigDict(extra='forbid')"),
    ])


def check_drift_report(f):
    train = pd.DataFrame({"a": [1.0, 2.0, 3.0, 4.0, 5.0], "b": [10.0, 11.0, 12.0, np.nan, 13.0]})
    same = pd.DataFrame({"a": [2.0, 3.0, 4.0], "b": [11.0, 12.0, 11.5]})
    shifted = pd.DataFrame({"a": [5.0, 6.0, 7.0, np.nan], "b": [10.0, 11.0, 12.0, 13.0]})

    def report(new):
        r = f(train, new, ["a", "b"])
        return {col: (round(float(r.loc[col, "shift_sd"]), 4), round(float(r.loc[col, "outside_range"]), 4), bool(r.loc[col, "flag"]))
                for col in ["a", "b"]}

    sd_a = train["a"].std()
    _scenarios("drift_report", [
        ("no drift: nothing flagged", lambda: report(same),
         {"a": (0.0, 0.0, False), "b": (round((11.5 - 11.5) / train["b"].std(), 4), 0.0, False)}, None),
        ("a shifted column is flagged; missing values ignored", lambda: report(shifted),
         {"a": (round((6.0 - 3.0) / sd_a, 4), round(2 / 3, 4), True), "b": (0.0, 0.0, False)},
         "(new mean - train mean) / train std, and the fraction of new values outside [train min, train max]"),
        ("indexed by column name", lambda: list(f(train, same, ["b", "a"]).index), ["b", "a"], None),
    ])


def check_nested_cv_score(f):
    X, y = load_breast_cancer(return_X_y=True, as_frame=True)
    X, y = X.iloc[:240, :8], y.iloc[:240]
    estimator = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    grid = {"logisticregression__C": [0.01, 1.0, 100.0]}
    outer = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)
    inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=1)

    reference_scores, reference_params = [], []
    for tr, te in outer.split(X, y):
        s = GridSearchCV(estimator, grid, cv=inner).fit(X.iloc[tr], y.iloc[tr])
        reference_scores.append(s.score(X.iloc[te], y.iloc[te]))
        reference_params.append(s.best_params_)

    result = {}

    def run():
        if "value" not in result:
            result["value"] = f(estimator, grid, X, y, outer, inner)
        return result["value"]

    _scenarios("nested_cv_score", [
        ("one score per outer fold, matching a reference", lambda: run()[0], np.array(reference_scores),
         "fit GridSearchCV on X.iloc[train_idx] only, then .score on the test indices"),
        ("best parameters from each fold", lambda: run()[1], reference_params, "collect search.best_params_ each time"),
        ("the estimator passed in stays unfitted", lambda: hasattr(estimator[-1], "coef_"), False,
         "GridSearchCV clones it; don't fit the original"),
    ])


def check_permutation_importance(f):
    class OnlyUsesA:
        def predict(self, X):
            return (X["a"].to_numpy() > 0).astype(int)

    rng_data = np.random.default_rng(0)
    X = pd.DataFrame({"a": rng_data.normal(size=200), "b": rng_data.normal(size=200), "c": rng_data.normal(size=200)})
    y = (X["a"] > 0).astype(int).to_numpy()

    def result():
        return f(OnlyUsesA(), X, y, np.random.default_rng(1), n_repeats=3)

    def unchanged():
        before = X.copy()
        f(OnlyUsesA(), X, y, np.random.default_rng(2), n_repeats=2)
        return X.equals(before)

    def repeats_used():
        calls = []

        class Counting(OnlyUsesA):
            def predict(self, X):
                calls.append(1)
                return super().predict(X)
        f(Counting(), X, y, np.random.default_rng(3), n_repeats=4)
        return len(calls)

    def reproducible():
        return f(OnlyUsesA(), X, y, np.random.default_rng(5)).equals(f(OnlyUsesA(), X, y, np.random.default_rng(5)))

    _scenarios("permutation_importance", [
        ("sorted, with the used feature first", lambda: list(result().index), ["a", "b", "c"], "sort_values(ascending=False)"),
        ("shuffling the used feature costs about half the accuracy", lambda: 0.35 < result()["a"] < 0.65, True,
         "baseline accuracy minus the mean accuracy with that column shuffled"),
        ("features the model ignores have exactly zero importance", lambda: (result()["b"], result()["c"]), (0.0, 0.0),
         "shuffle only one column at a time, on a copy"),
        ("X is not modified", unchanged, True, "shuffle a copy: shuffled = X.copy()"),
        ("one baseline call plus n_repeats per column", repeats_used, 1 + 4 * 3, None),
        ("same seed, same result", reproducible, True, "use the rng passed in"),
    ])


def check_create_app(f):
    class FakeModel:
        """P(male) rises with body mass. Classes in the unusual order, male first."""
        classes_ = np.array(["male", "female"])

        def predict_proba(self, df):
            p_male = 1 / (1 + np.exp(-(df["body_mass_g"].to_numpy() - 4200) / 300))
            return np.column_stack([p_male, 1 - p_male])

    def penguin(mass, species="Adelie"):
        return {"species": species, "bill_length_mm": 40.0, "bill_depth_mm": 18.0, "flipper_length_mm": 195.0, "body_mass_g": mass}

    def expected(mass, margin=0.0):
        p = 1 / (1 + math.exp(-(mass - 4200) / 300))
        sex = "unsure" if abs(p - 0.5) < margin else ("male" if p >= 0.5 else "female")
        return {"sex": sex, "p_male": round(p, 3)}

    def client(margin=0.0):
        return TestClient(f(FakeModel(), margin=margin))

    def single(mass, margin=0.0):
        r = client(margin).post("/predict", json=penguin(mass))
        return (r.status_code, r.json())

    def batch(masses, margin=0.0):
        r = client(margin).post("/predict/batch", json={"penguins": [penguin(m) for m in masses]})
        return (r.status_code, r.json() if r.status_code == 200 else None)

    _scenarios("create_app", [
        ("GET /health", lambda: client().get("/health").json(), {"status": "ok"}, None),
        ("a heavy penguin is male", lambda: single(5200), (200, expected(5200)),
         "find the male column with list(model.classes_).index('male')"),
        ("a light penguin is female", lambda: single(3300), (200, expected(3300)), None),
        ("near 0.5 with a margin: unsure", lambda: single(4250, margin=0.1), (200, expected(4250, 0.1)),
         "'unsure' when abs(p_male - 0.5) < margin"),
        ("invalid input is rejected with 422", lambda: client().post("/predict", json=penguin(37)).status_code, 422,
         "use PenguinIn as the argument type"),
        ("batch: one result per penguin, in order", lambda: batch([5200, 3300, 4250], margin=0.1),
         (200, {"predictions": [expected(5200, 0.1), expected(3300, 0.1), expected(4250, 0.1)]}), None),
        ("an empty batch is rejected", lambda: batch([])[0], 422, "Field(min_length=1, max_length=100)"),
        ("more than 100 penguins is rejected", lambda: batch([4000] * 101)[0], 422, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `snake_case`

In [2]:
import re


def snake_case(name: str) -> str:
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)     # studyName -> study_Name
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)               # spaces, brackets, slashes -> _
    return name.strip("_").lower()

In [3]:
check_snake_case(snake_case)

PASS  all 7 tests passed for snake_case


Two regular expressions, applied in order:

1. `([a-z0-9])([A-Z])` finds a lowercase letter or digit followed by a capital, and puts `_` between them. In `"Individual ID"`, the `D` after `I` doesn't match (capital followed by capital), so `ID` stays one word.
2. `[^A-Za-z0-9]+` matches any run of other characters, so `" (o/oo)"` becomes a single `_` rather than four.

The camel-case step has to come **first**, while the capitals are still there to detect. After `.lower()` there's no way to tell where the words were.

Standardising names at load time, in one place, means the rest of the code never has to deal with `"Culmen Length (mm)"` again. In a pipeline that runs every day, it also protects you from a supplier who changes `"Body Mass (g)"` to `"Body mass (g)"`: both map to the same name.

### Exercise 2: `clean_species`

In [4]:
SPECIES = ["Adelie", "Chinstrap", "Gentoo"]


def clean_species(series: pd.Series) -> pd.Series:
    if series.isna().any():
        raise ValueError(f"{series.isna().sum()} missing species values")
    first_word = series.astype(str).str.strip().str.split().str[0].str.capitalize()
    bad = sorted(set(series[~first_word.isin(SPECIES)]))
    if bad:
        raise ValueError(f"unknown species: {bad}")
    return first_word

In [5]:
check_clean_species(clean_species)

PASS  all 4 tests passed for clean_species


The `.str` accessor chains string methods over a whole column: strip, split into words, take the first (`.str[0]` indexes into each list), capitalise.

The error lists the **original** values, not the cleaned ones. If a file contains `"Emperor penguin (Aptenodytes forsteri)"`, the person reading the error needs to see that, not `"Emperor"`.

Refusing to guess is deliberate. A typo like `"Gentooo"` could be mapped to Gentoo with fuzzy matching, but a validation step that silently "fixes" data is how wrong labels get into a training set. Fail, look at the value, and if it's a real variant, add it to the rules explicitly.

### Exercise 3: `clean_sex`

In [6]:
def clean_sex(series: pd.Series) -> pd.Series:
    cleaned = series.str.strip().str.lower()           # missing stays missing
    return cleaned.where(cleaned.isin(["male", "female"]))

In [7]:
check_clean_sex(clean_sex)

PASS  all 3 tests passed for clean_sex


`.where(condition)` keeps values where the condition is true and replaces the rest with missing. That turns "anything that isn't exactly male or female" into missing in one step, which is safer than listing every placeholder you've seen so far: next year's file might use `"?"`.

The `.str` methods pass missing values through untouched, which is what you want. Be careful with the tempting alternative, `series.astype(str)`. In pandas 2, which Colab uses, it turns a missing value into the **text** `"nan"`, which then looks like a real value to every later step. (pandas 3 changed this and keeps missing values missing, but code that depends on the version is fragile.)

For the target column of a model, missing means "can't be used for training or evaluation". The capstone drops those rows, and **counts and reports** them.

### Exercise 4: `out_of_range`

In [8]:
def out_of_range(df: pd.DataFrame, limits: dict) -> pd.DataFrame:
    problems = []
    for order, (column, (low, high)) in enumerate(limits.items()):
        values = df[column]
        bad = values.notna() & ((values < low) | (values > high))
        for row, value in values[bad].items():
            problems.append({"row": row, "column": column, "value": value, "_order": order})
    if not problems:
        return pd.DataFrame(columns=["row", "column", "value"])
    report = pd.DataFrame(problems).sort_values(["row", "_order"], kind="stable")
    return report.drop(columns="_order").reset_index(drop=True)

In [9]:
check_out_of_range(out_of_range)

PASS  all 4 tests passed for out_of_range


A report like this is more useful than a yes/no check: it says exactly which rows to look at. In practice, out-of-range values have causes worth knowing. A body mass of 37 is probably kilograms entered in a grams column. A bill depth of 175 might be a flipper length in the wrong column. A 0 often means "not measured".

Comparisons with `NaN` are always `False`, so `values < low` would skip missing values anyway, but the explicit `notna()` makes the intention clear to a reader.

Sorting by the position of the column in `limits` (rather than alphabetically) keeps the report in the order the person who wrote the limits thought about them. `kind="stable"` guarantees rows with equal keys keep their original order.

## Medium

### Exercise 5: `make_preprocessor`

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor(numeric: list, categorical: list) -> ColumnTransformer:
    return ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), list(numeric)),
        ("categorical", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                      OneHotEncoder(handle_unknown="ignore", sparse_output=False)), list(categorical)),
    ], remainder="drop")

In [11]:
check_make_preprocessor(make_preprocessor)

PASS  all 5 tests passed for make_preprocessor


Returning an **unfitted** object is the important design choice. It goes inside a pipeline, and cross-validation fits it separately on each training fold. If the function fitted it on the whole dataset, the medians and means would include the test rows: a small leak, but a leak.

The median is preferred to the mean for filling numeric gaps because it isn't pulled around by outliers, which is exactly when you'd least want your fill value to move.

`handle_unknown="ignore"` matters most at prediction time. A model that crashes when a new island appears in the data is fragile; one that encodes it as "none of the islands I know" keeps working, and the drift check (exercise 9) can flag the change.

`remainder="drop"` is the default, but writing it makes the behaviour obvious: an ID column or a comments field never sneaks into the model.

One version trap: `SimpleImputer` looks for `np.nan` by default. Data read with `pd.read_csv` uses `NaN` for gaps, so that works, but a Python `None` typed into a DataFrame by hand isn't treated as missing by older scikit-learn versions, and becomes a category of its own.

### Exercise 6: `compare_models`

In [12]:
from sklearn.model_selection import cross_val_score


def compare_models(models: dict, X, y, cv) -> pd.DataFrame:
    rows = {}
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv)
        rows[name] = {"mean": scores.mean(), "std": scores.std()}
    return pd.DataFrame.from_dict(rows, orient="index").sort_values("mean", ascending=False)

In [13]:
check_compare_models(compare_models)

PASS  all 3 tests passed for compare_models


Passing the **same** `cv` object to every model matters. A `StratifiedKFold` with a fixed `random_state` produces the same folds every time it's used, so all models are tested on identical splits, and the differences between them aren't just differences in luck. (Passing `cv=5` also gives the same folds each time, but without shuffling.)

`cross_val_score` clones each model before fitting, so the estimators in `models` stay unfitted and can be reused.

The standard deviation is what keeps the comparison honest. A model that's 0.5 points ahead with a fold spread of 3 points hasn't been shown to be better. The capstone uses a simple rule: among models within one standard deviation of the best, choose the simplest.

### Exercise 7: `selective_accuracy`

In [14]:
def selective_accuracy(y_true, p, margin: float) -> dict:
    y_true = np.asarray(y_true)
    p = np.asarray(p, dtype=float)
    covered = np.abs(p - 0.5) >= margin
    n_covered = int(covered.sum())
    if n_covered == 0:
        return {"coverage": 0.0, "accuracy": float("nan"), "n_covered": 0}
    correct = (p[covered] >= 0.5).astype(int) == y_true[covered]
    return {"coverage": n_covered / len(p), "accuracy": float(correct.mean()), "n_covered": n_covered}

In [15]:
check_selective_accuracy(selective_accuracy)

PASS  all 5 tests passed for selective_accuracy


The trade-off this measures is coverage against accuracy: as the margin grows, the model answers less often but is right more often on the cases it does answer. Plotting one against the other, for a range of margins, shows the options to whoever will use the model, who is usually better placed than you to decide how many "unsure" answers are acceptable.

The margin must be chosen on **validation** predictions (in the capstone, out-of-fold predictions from cross-validation), never on the test set. Otherwise the reported accuracy is tuned to the test data, and it's leakage again.

Returning `nan` when nothing is covered, rather than 0 or raising, is honest: accuracy is undefined there, not zero.

### Exercise 8: `PenguinIn`

In [16]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field


class PenguinIn(BaseModel):
    model_config = ConfigDict(extra="forbid")

    species: Literal["Adelie", "Chinstrap", "Gentoo"]
    bill_length_mm: float = Field(ge=25, le=70)
    bill_depth_mm: float = Field(ge=10, le=25)
    flipper_length_mm: float = Field(ge=160, le=240)
    body_mass_g: float = Field(ge=2000, le=7000)

In [17]:
check_PenguinIn(PenguinIn)

PASS  all 8 tests passed for PenguinIn


The limits are wider than anything in the training data, on purpose. They're a **sanity** check (no 20 kg penguins, no grams typed into the millimetres field), not a statement about what the model has seen. Values that are plausible but unusual are a different situation: they should get a prediction **with a warning**, which the capstone API adds using the training ranges.

`extra="forbid"` catches a whole class of client bugs. Without it, a request with `"body_mass"` instead of `"body_mass_g"` would fail only because the real field is missing; with an optional field, it would silently use the default. Pydantic's default is to ignore extra fields.

Pydantic converts compatible input: the string `"39.1"` becomes the float 39.1. That's convenient for clients that send everything as text, and the check confirms it. If you need to reject strings, Pydantic has a `strict` mode.

### Exercise 9: `drift_report`

In [18]:
def drift_report(train: pd.DataFrame, new: pd.DataFrame, columns: list) -> pd.DataFrame:
    rows = {}
    for column in columns:
        reference, incoming = train[column].dropna(), new[column].dropna()
        shift = (incoming.mean() - reference.mean()) / reference.std()
        outside = ((incoming < reference.min()) | (incoming > reference.max())).mean()
        rows[column] = {"shift_sd": shift, "outside_range": outside,
                        "flag": bool(abs(shift) > 0.5 or outside > 0.05)}
    return pd.DataFrame.from_dict(rows, orient="index")

In [19]:
check_drift_report(drift_report)

PASS  all 3 tests passed for drift_report


Measuring the shift in units of the training standard deviation makes columns comparable: a 100 g change in body mass and a 1 mm change in bill depth can be judged on the same scale.

The two statistics catch different problems. **`shift_sd`** catches a population that has moved as a whole: new equipment that reads slightly high, or a new site with bigger birds. **`outside_range`** catches individual values the model has never seen anything like, where its predictions are extrapolations. A unit error usually shows up in both.

The thresholds (0.5 standard deviations, 5%) are starting points, not rules. In practice you'd tune them so that real problems are flagged without crying wolf every week. More formal tests exist (Kolmogorov-Smirnov, population stability index), but a simple, explainable report like this one catches most real incidents.

## Hard

### Exercise 10: `nested_cv_score`

In [20]:
from sklearn.model_selection import GridSearchCV


def nested_cv_score(estimator, param_grid: dict, X, y, outer_cv, inner_cv):
    scores, best_params = [], []
    for train_idx, test_idx in outer_cv.split(X, y):
        search = GridSearchCV(estimator, param_grid, cv=inner_cv)
        search.fit(X.iloc[train_idx], y.iloc[train_idx])
        scores.append(search.score(X.iloc[test_idx], y.iloc[test_idx]))
        best_params.append(search.best_params_)
    return np.array(scores), best_params

In [21]:
check_nested_cv_score(nested_cv_score)

PASS  all 3 tests passed for nested_cv_score


The lesson did this with one line, `cross_val_score(search, X, y, cv=outer)`. Writing the loop shows what that line does, and it returns something the one-liner doesn't: **which parameters won in each fold**.

That list is worth looking at. If the same value wins in every fold, the choice is stable and the final model's setting is trustworthy. If the winner jumps around (`C = 0.1`, then 100, then 1), the scores for different settings are within noise of each other, and the exact choice doesn't matter much. The capstone sees exactly that.

`GridSearchCV` clones the estimator it's given, so the same `estimator` object can be reused across folds safely. With `refit=True` (the default), `search.score` uses the best model refitted on the whole training part of the fold, which is what you'd do with the final model too.

### Exercise 11: `permutation_importance`

In [22]:
def permutation_importance(model, X: pd.DataFrame, y, rng, n_repeats: int = 5) -> pd.Series:
    y = np.asarray(y)
    baseline = np.mean(model.predict(X) == y)
    drops = {}
    for column in X.columns:
        scores = []
        for _ in range(n_repeats):
            shuffled = X.copy()
            shuffled[column] = rng.permutation(shuffled[column].to_numpy())
            scores.append(np.mean(model.predict(shuffled) == y))
        drops[column] = baseline - np.mean(scores)
    return pd.Series(drops).sort_values(ascending=False)

In [23]:
check_permutation_importance(permutation_importance)

PASS  all 6 tests passed for permutation_importance


Shuffling keeps the column's distribution exactly the same (same values, same mean, same spread) but destroys its relationship with everything else. So the drop in accuracy measures how much the model **uses** that relationship.

Three cautions when reading the results:

- It measures what the **model** relies on, not what's true about the world. A model can ignore a useful feature if another one carries the same information.
- **Correlated features share importance.** Shuffle one of two near-duplicate columns and the model can lean on the other, so both look unimportant, even though removing both would hurt a lot.
- Compute it on **held-out** data when you can. On training data it shows what the model memorised as well as what it learned.

scikit-learn has this as `sklearn.inspection.permutation_importance`, with the same idea and more options.

### Exercise 12: `create_app`

In [24]:
from fastapi import FastAPI


class BatchIn(BaseModel):
    penguins: list[PenguinIn] = Field(min_length=1, max_length=100)


def create_app(model, margin: float = 0.0) -> FastAPI:
    male_column = list(model.classes_).index("male")
    app = FastAPI(title="Penguin sex classifier")

    def predict_many(penguins):
        frame = pd.DataFrame([p.model_dump() for p in penguins])
        p_male = model.predict_proba(frame)[:, male_column]
        results = []
        for p in p_male:
            if abs(p - 0.5) < margin:
                sex = "unsure"
            else:
                sex = "male" if p >= 0.5 else "female"
            results.append({"sex": sex, "p_male": round(float(p), 3)})
        return results

    @app.get("/health")
    def health():
        return {"status": "ok"}

    @app.post("/predict")
    def predict(penguin: PenguinIn):
        return predict_many([penguin])[0]

    @app.post("/predict/batch")
    def predict_batch(batch: BatchIn):
        return {"predictions": predict_many(batch.penguins)}

    return app

In [25]:
check_create_app(create_app)

PASS  all 8 tests passed for create_app


**A factory function** (`create_app(model)`) instead of a module-level `app` makes the API testable: each test can build an app around a small fake model, and the real service builds one around the loaded bundle. It's the same idea as `main(argv=None)` in Module 10.

**One code path for both endpoints.** The single-penguin endpoint calls the batch logic with a list of one, so the two can't drift apart, and there's only one place where the "unsure" rule lives.

**`classes_` order.** scikit-learn sorts class labels, so for `"female"`/`"male"` the male probability is column 1. But a model trained with labels 0/1, or by another library, might differ, and assuming column 1 would silently return P(female) labelled as P(male). The check uses a fake model with the classes reversed to catch exactly that.

**Batch limits** protect the service: without a maximum, one request with a million penguins could tie it up for everyone. `min_length=1` turns an empty request, almost certainly a client bug, into a clear error.

`round(float(p), 3)` converts the NumPy number to a Python float, which is what JSON needs; FastAPI would usually manage the conversion itself, but being explicit avoids surprises.

## What's next

The capstone in `04_capstone.ipynb` builds the whole system: raw data, cleaning and validation, honest model selection, a saved model bundle with a model card, and a tested FastAPI service.